# Test — Bronze CSV COPY INTO
Runs as a task in `bronze_ingestion_job` (depends on `ingest_streets_copyinto`).

In [0]:
import unittest
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.testing.utils import assertSchemaEqual

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("bronze_schema", "bronze", "2. Bronze Schema")

CATALOG = dbutils.widgets.get("catalog_name")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.streets_csv_copyinto"

EXPECTED_SCHEMA = StructType([
    StructField("noise", StringType(), True),
    StructField("pollution", StringType(), True),
    StructField("date", StringType(), True),
    StructField("light", StringType(), True),
    StructField("raining", StringType(), True),
    StructField("street_id", StringType(), True),
    StructField("load_dt", TimestampType(), True),
    StructField("source", StringType(), True),
])


class BronzeCopyIntoTests(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.df = spark.table(TABLE)
        cls.count = cls.df.count()

    def test_table_has_rows(self):
        self.assertGreater(self.count, 0, f"{TABLE} is empty.")

    def test_schema_matches_expected(self):
        assertSchemaEqual(self.df.schema, EXPECTED_SCHEMA)

    def test_audit_columns_populated(self):
        missing = self.df.filter("load_dt IS NULL OR source IS NULL").count()
        self.assertEqual(missing, 0, f"{missing} rows missing load_dt/source.")

    def test_source_file_is_correct_chunk(self):
        sources = {r["source"] for r in self.df.select("source").distinct().collect()}
        self.assertEqual(sources, {"streets_chunk_1.csv"},
                          f"Expected only streets_chunk_1.csv as source, found: {sources}")

    def test_table_and_columns_have_descriptions(self):
        """PDF requirement: 'descriptions/metadata to make data discoverable.'"""
        tbl_comment = spark.sql(f"DESCRIBE TABLE EXTENDED {TABLE}") \
            .filter("col_name = 'Comment'").collect()
        self.assertTrue(tbl_comment and tbl_comment[0]["data_type"],
                         "Table is missing a COMMENT.")


# COMMAND ----------

if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(BronzeCopyIntoTests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Bronze COPY INTO tests FAILED — see output above.")
